# Retrieval pipeline — the "R" in RAG

In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os
import chromadb

In [2]:
env_path = Path(r"C:\Users\jasme\Documents\rag-knowledge-assistant\.env")
load_dotenv(dotenv_path=env_path, override=True)



True

In [3]:
# Create ChromaDB client
chroma_client = chromadb.Client()

# Create a collection (like a table in a normal database)
collection = chroma_client.create_collection(
    name="day2_test",
    metadata={"description": "My first vector collection"}
)

print("✅ ChromaDB collection created!")
print(f"Collection name: {collection.name}")

✅ ChromaDB collection created!
Collection name: day2_test


In [4]:
# Add documents to ChromaDB
# ChromaDB automatically creates embeddings for each document
collection.add(
    documents=[
        "Retrieval-Augmented Generation improves AI accuracy using real documents.",
        "Machine learning models learn patterns from training data.",
        "ChromaDB is a vector database that stores embeddings for semantic search.",
        "Python is the most popular language for AI and data science.",
        "LLMs can hallucinate and produce confident but wrong answers.",
        "RAG systems retrieve relevant context before generating a response.",
        "Neural networks are inspired by the human brain structure.",
        "FastAPI is a modern Python framework for building REST APIs.",
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6", "doc7", "doc8"]
)

print("✅ 8 documents added to ChromaDB!")
print(f"Total documents in collection: {collection.count()}")

C:\Users\jasme\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 11.2MiB/s]


✅ 8 documents added to ChromaDB!
Total documents in collection: 8


In [5]:
# In-memory (what you have now — for learning)
client = chromadb.Client()

# Persistent (what we use in production — Phase 2)
client = chromadb.PersistentClient(path="./chroma_db")

In [9]:
# Peek at what is stored inside
results = collection.peek()

print(f"Total documents: {collection.count()}")
print(f"\nFirst 3 document IDs: {results['ids'][:3]}")
print(f"\nFirst 3 documents:")
for i, doc in enumerate(results['documents'][:3]):
    print(f"  {i+1}. {doc}")

Total documents: 8

First 3 document IDs: ['doc1', 'doc2', 'doc3']

First 3 documents:
  1. Retrieval-Augmented Generation improves AI accuracy using real documents.
  2. Machine learning models learn patterns from training data.
  3. ChromaDB is a vector database that stores embeddings for semantic search.


## Next Step — Semantic Search

In [11]:
# Correct relevance calculation
query = "How do AI systems avoid making things up?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: '{query}'")
print(f"\nTop 3 most relevant documents:\n")
for i, (doc, distance) in enumerate(zip(
    results['documents'][0],
    results['distances'][0]
)):
    # Correct formula — lower distance = higher similarity
    relevance = round((1 / (1 + distance)) * 100, 1)
    print(f"  #{i+1} (relevance: {relevance}%)")
    print(f"       {doc}")
    print()

Query: 'How do AI systems avoid making things up?'

Top 3 most relevant documents:

  #1 (relevance: 48.3%)
       Retrieval-Augmented Generation improves AI accuracy using real documents.

  #2 (relevance: 44.9%)
       Neural networks are inspired by the human brain structure.

  #3 (relevance: 44.8%)
       Machine learning models learn patterns from training data.



# Rewriting the code for better result

__Upgrading your embedding model for better result__: As relevance was very poor

Old model (default):
- Very basic, general purpose
- ~50MB, low quality embeddings
- Good for: quick demos only

New model (all-MiniLM-L6-v2):
- Trained specifically for sentence similarity
- 384-dimension embeddings
- Used in real production RAG systems
- Good for: actual semantic search

In [3]:
from chromadb.utils import embedding_functions

# Upgrade to a proper sentence embedding model
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\jasme\anaconda3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4468.26it/s]


In [4]:
# Create ChromaDB client
chroma_client = chromadb.Client()

In [5]:
# Create a NEW collection using the better model
better_collection = chroma_client.create_collection(
    name="day2_upgraded",
    embedding_function=sentence_transformer_ef
)

In [6]:
# Add the same 8 documents
better_collection.add(
    documents=[
        "Retrieval-Augmented Generation improves AI accuracy using real documents.",
        "Machine learning models learn patterns from training data.",
        "ChromaDB is a vector database that stores embeddings for semantic search.",
        "Python is the most popular language for AI and data science.",
        "LLMs can hallucinate and produce confident but wrong answers.",
        "RAG systems retrieve relevant context before generating a response.",
        "Neural networks are inspired by the human brain structure.",
        "FastAPI is a modern Python framework for building REST APIs.",
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6", "doc7", "doc8"]
)

print("✅ Upgraded collection ready!")
print(f"Documents stored: {better_collection.count()}")

✅ Upgraded collection ready!
Documents stored: 8


The upgraded embedding model is ready. Now let's see the real difference it makes.

In [7]:
query = "How do AI systems avoid making things up?"

results = better_collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: '{query}'")
print(f"\nTop 3 most relevant documents (UPGRADED model):\n")
for i, (doc, distance) in enumerate(zip(
    results['documents'][0],
    results['distances'][0]
)):
    relevance = round((1 / (1 + distance)) * 100, 1)
    print(f"  #{i+1} (relevance: {relevance}%)")
    print(f"       {doc}")
    print()

Query: 'How do AI systems avoid making things up?'

Top 3 most relevant documents (UPGRADED model):

  #1 (relevance: 65.1%)
       Retrieval-Augmented Generation improves AI accuracy using real documents.

  #2 (relevance: 62.0%)
       Neural networks are inspired by the human brain structure.

  #3 (relevance: 61.8%)
       Machine learning models learn patterns from training data.



# Rewriting the code for using PDF

__Step 1__ — Load the PDF and see what's inside

In [8]:
from pypdf import PdfReader

# Load your PDF
pdf_path = r"C:\Users\jasme\Documents\rag-knowledge-assistant\data\test_for_rag.pdf"
reader = PdfReader(pdf_path)

print(f"✅ PDF loaded successfully!")
print(f"Total pages: {len(reader.pages)}")

# Extract text from the first page to check it works
first_page_text = reader.pages[0].extract_text()
print(f"\n--- First 500 characters of page 1 ---")
print(first_page_text[:500])

✅ PDF loaded successfully!
Total pages: 14

--- First 500 characters of page 1 ---
PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within
PDFs—including text, images, vector diagrams, gr


__Step 2__ — Extract ALL pages and combine them

In [9]:
# Extract text from all pages
all_text = ""
for page in reader.pages:
    all_text += page.extract_text() + "\n"

print(f"✅ Extracted text from {len(reader.pages)} pages")
print(f"Total characters: {len(all_text)}")
print(f"\n--- Preview (first 300 chars) ---")
print(all_text[:300])

✅ Extracted text from 14 pages
Total characters: 42145

--- Preview (first 300 chars) ---
PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a


__Step 3__ — Split into chunks (the most important step)

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split the text into chunks
chunks = splitter.split_text(all_text)

print(f"✅ Document split into {len(chunks)} chunks")
print(f"\n--- First chunk ---")
print(chunks[0])
print(f"\n--- Second chunk ---")
print(chunks[1])

✅ Document split into 93 chunks

--- First chunk ---
PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within

--- Second chunk ---
PDFs—including text, images, vector diagrams, graphs, and tables—poses unique
challenges for existing QA systems primarily designed for textual content. We seek
to develop a comprehensive RAG-based QA system that will effectively address
complex multi-modal questions, where several data types are put together in the
question. This is mainly achieved by refining approaches toward processing and
integrating non-textual elements in PDFs into the RAG framework to derive pr

What you just achieved?

14 pages → 93 chunks of ~500 characters each

__Step 4__ — Store all 93 chunks in ChromaDB with embeddings

In [17]:
# Create a NEW collection for the real PDF
pdf_collection = chroma_client.create_collection(
    name="rag_paper",
    embedding_function=sentence_transformer_ef
)


In [18]:
pdf_collection

Collection(name=rag_paper)

In [19]:
# Generate IDs for each chunk
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

In [20]:
chunk_ids

['chunk_0',
 'chunk_1',
 'chunk_2',
 'chunk_3',
 'chunk_4',
 'chunk_5',
 'chunk_6',
 'chunk_7',
 'chunk_8',
 'chunk_9',
 'chunk_10',
 'chunk_11',
 'chunk_12',
 'chunk_13',
 'chunk_14',
 'chunk_15',
 'chunk_16',
 'chunk_17',
 'chunk_18',
 'chunk_19',
 'chunk_20',
 'chunk_21',
 'chunk_22',
 'chunk_23',
 'chunk_24',
 'chunk_25',
 'chunk_26',
 'chunk_27',
 'chunk_28',
 'chunk_29',
 'chunk_30',
 'chunk_31',
 'chunk_32',
 'chunk_33',
 'chunk_34',
 'chunk_35',
 'chunk_36',
 'chunk_37',
 'chunk_38',
 'chunk_39',
 'chunk_40',
 'chunk_41',
 'chunk_42',
 'chunk_43',
 'chunk_44',
 'chunk_45',
 'chunk_46',
 'chunk_47',
 'chunk_48',
 'chunk_49',
 'chunk_50',
 'chunk_51',
 'chunk_52',
 'chunk_53',
 'chunk_54',
 'chunk_55',
 'chunk_56',
 'chunk_57',
 'chunk_58',
 'chunk_59',
 'chunk_60',
 'chunk_61',
 'chunk_62',
 'chunk_63',
 'chunk_64',
 'chunk_65',
 'chunk_66',
 'chunk_67',
 'chunk_68',
 'chunk_69',
 'chunk_70',
 'chunk_71',
 'chunk_72',
 'chunk_73',
 'chunk_74',
 'chunk_75',
 'chunk_76',
 'chunk_7

In [21]:
# Add all chunks to ChromaDB
pdf_collection.add(
    documents=chunks,
    ids=chunk_ids
)

print(f"✅ Stored {pdf_collection.count()} chunks in ChromaDB!")

✅ Stored 93 chunks in ChromaDB!


__What to expect__

With 93 real chunks from a real paper, the relevance scores should be much more spread out — likely something like 75%, 60%, 45% instead of 65/62/61. And chunk 1 from earlier (which literally discusses "challenges...for existing QA systems") should rank very high.

__Step 5__ — the actual search query:

In [23]:
query = "What challenges does this paper address with PDFs?"

results = pdf_collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: '{query}'\n")
for i, (chunk, distance) in enumerate(zip(
    results['documents'][0],
    results['distances'][0]
)):
    relevance = round((1 / (1 + distance)) * 100, 1)
    print(f"#{i+1} (relevance: {relevance}%)")
    print(chunk)
    print("-" * 60)

Query: 'What challenges does this paper address with PDFs?'

#1 (relevance: 66.2%)
academia and industries due to the data richness in PDF, from plain text, tables to high resolution
images and intricate vector graphics, presenting an opportunity and a challenge at the same time.
Traditional RAG-based QA systems focus primarily on text Lin [2024], Ma et al. [2023], Siriwardhana
et al. [2023] while non-textual elements such as images, charts, tables and diagrams within PDFs
are not thoroughly explored. Our objective is to address this gap by developing a comprehensive
------------------------------------------------------------
#2 (relevance: 65.4%)
types of content from PDFs, including text, images, and layouts. By default, it uses the ‘Surya’ OCR
engine for efficient and accurate text recognition. The tool is configured to run OCR on all pages of a
PDF, even if some text can be directly extracted, ensuring comprehensive text recognition across the
entire document.
In addition to text 

# Where data is stored?

Everything i built today — my ChromaDB collections, my 93 chunks, my embeddings — exists only in my computer's RAM (memory) while the notebook kernel is running.

In [24]:
import chromadb

# Check what type of client you're using
print(type(chroma_client))
print(chroma_client)

<class 'chromadb.api.client.Client'>


To make it PERMANENT — save to disk

In [25]:
# Create a PERSISTENT client — saves to disk
persistent_client = chromadb.PersistentClient(
    path=r"C:\Users\jasme\Documents\rag-knowledge-assistant\chroma_db"
)

print(type(persistent_client))
print("✅ Persistent client created!")

<class 'chromadb.api.client.Client'>
✅ Persistent client created!


In [26]:
pdf_collection_persistent = persistent_client.create_collection(
    name="rag_paper",
    embedding_function=sentence_transformer_ef
)

chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

pdf_collection_persistent.add(
    documents=chunks,
    ids=chunk_ids
)

print(f"✅ Stored {pdf_collection_persistent.count()} chunks PERMANENTLY!")

✅ Stored 93 chunks PERMANENTLY!


In [27]:
print(f"Documents in persistent collection: {pdf_collection_persistent.count()}")

Documents in persistent collection: 93


Several ways to look inside your chroma_db — 

1. from simplest (Python)
2. to most visual (GUI tools)

In [29]:
# List all collections in your database
collections = persistent_client.list_collections()
print("Collections in your database:")
for col in collections:
    print(f"  - {col.name}")

Collections in your database:
  - rag_paper


In [30]:
# Peek at the first 3 entries — see actual stored data
data = collection.peek(limit=3)
print(f"\nIDs: {data['ids']}")
print(f"\nFirst document text:\n{data['documents'][0]}")
print(f"\nFirst embedding (first 10 numbers of 384):\n{data['embeddings'][0][:10]}")


IDs: ['chunk_0', 'chunk_1', 'chunk_2']

First document text:
PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within

First embedding (first 10 numbers of 384):
[-9.07424688e-02  2.41668504e-02 -1.28467590e-01  5.56564778e-02
 -1.39999986e-02  4.94681783e-02 -9.18721980e-06  1.51448755e-03
 -2.34669214e-03 -1.96974855e-02]


In [31]:
# Try adding chunk_0 again with the SAME id
try:
    collection.add(
        documents=[chunks[0]],
        ids=["chunk_0"]  # same ID as before!
    )
    print("Added again")
except Exception as e:
    print(f"Error: {e}")

Added again


In [32]:
print(f"Total count: {collection.count()}")

# Get chunk_0 specifically
result = collection.get(ids=["chunk_0"])
print(f"\nNumber of entries with id 'chunk_0': {len(result['ids'])}")
print(f"Documents returned: {result['documents']}")

Total count: 93

Number of entries with id 'chunk_0': 1
Documents returned: ['PDF Retrieval Augmented Question Answering\nThi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen\nSaarland University\n{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de\nAbstract\nThis paper presents an advancement in Question-Answering (QA) systems using\na Retrieval Augmented Generation (RAG) framework to enhance information\nextraction from PDF files. Recognizing the richness and diversity of data within']
